# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muneeb-th/ML-Assignment-1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Muneeb-th/ML-Assignment-1/main/data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

for col in ["days_since_last_update", "impressions_90d", "ctr", "avg_position", "word_count"]:
    print(f"\n{col}:")
    print(df[col].describe())


days_since_last_update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

impressions_90d:
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64

ctr:
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64

avg_position:
count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64

word_count:
count    22301.000000
mean      3107.760325
std       1452.382598
min      

Several fields show heavy tails: impressions_90d and word_count both
have long right tails (a few very high-traffic/long pages skew the
mean well above the median). ctr has some implausible values above 1.0,
consistent with a data artifact noted in earlier notebooks — treated as
noise, not signal.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal #1 (staleness): stale pages (180+ days since update) show a
47.1% decline rate vs 54.3% for non-stale pages. Verdict: OPPOSITE.

Signal #2 (low CTR at position): pages with below-median CTR for their
position show a 59.3% decline rate vs 58.3% for others, barely above
the 54.2% base rate. Verdict: MIXED.

Signal #3 (impressions volume): testing whether higher-traffic pages
decline less often, as a new third check.

In [6]:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

print("Signal 1 - Staleness:")
print(df.groupby(stale)["is_declining"].agg(["mean", "count"]))

df["median_ctr_for_tier"] = df.groupby("position_tier")["ctr"].transform("median")
low_ctr = (df["ctr"] < df["median_ctr_for_tier"]).astype(int)
print("\nSignal 2 - Low CTR at position:")
print(df.groupby(low_ctr)["is_declining"].agg(["mean", "count"]))

high_volume = (df["impressions_90d"] >= df["impressions_90d"].median()).astype(int)
print("\nSignal 3 - High volume (above median impressions):")
print(df.groupby(high_volume)["is_declining"].agg(["mean", "count"]))

Signal 1 - Staleness:
                            mean  count
days_since_last_update                 
0                       0.542480  29826
1                       0.471264    174

Signal 2 - Low CTR at position:
       mean  count
0  0.502630  16921
1  0.593088  13079

Signal 3 - High volume (above median impressions):
                     mean  count
impressions_90d                 
0                0.490095  14993
1                0.593989  15007


Signal 3 - High volume (above median impressions):
                     mean  count
impressions_90d                 
0                0.490095  14993
1                0.593989  15007

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Picking staleness, which underlies FlyRank's stale_visible_page flag.
The rule's assumption is "stale = needs review because it's likely
declining." The data does NOT support this assumption — staleness is
OPPOSITE signal (see Signal #1 above). This matches the finding from
Week 4's baseline notebook.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team using staleness alone as a review trigger would be
chasing the wrong pages — stale pages in this portfolio are not more
likely to be declining. Traffic volume and position are more useful
starting points for a review queue than page age alone.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.